In [49]:
print("안녕하세요!")

안녕하세요!


In [50]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas matplotlib seaborn missingno -q

import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno   # 결측 패턴 시각화 전용 라이브러리 (Part 3에서 사용)

warnings.filterwarnings("ignore")

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (운영체제별 분기)
system = platform.system()
if system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.4.6
pandas: 3.0.3


In [51]:
%pip install missingno


Note: you may need to restart the kernel to use updated packages.


In [52]:
# ─────────────────────────────────────────────
# 모두마켓 데이터 생성 (D+003 버전)
# ─────────────────────────────────────────────
np.random.seed(42)

# 1) 고객(customers)
n_customers = 300
customers = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_customers + 1)],
    "age": np.random.normal(35, 9, n_customers).round().astype(int),
    "gender": np.random.choice(["M", "F"], n_customers),
    "region": np.random.choice(["서울", "경기", "부산", "인천", "대구"], n_customers),
    "membership": np.random.choice(
        ["basic", "premium", "vip"], n_customers, p=[0.6, 0.3, 0.1]
    ),
    "income": np.random.choice([2500, 3500, 4500, 6000, 8500], n_customers).astype(float),
})

# 나이 오염
customers.loc[5, "age"] = 999
customers.loc[10, "age"] = -3
customers.loc[15, "age"] = 120

# 성별 결측
customers.loc[[20, 21, 22, 70, 120], "gender"] = np.nan

# 고소득 고객의 소득 결측
high_income_mask = customers["income"] >= 6000
high_idx = customers[high_income_mask].sample(frac=0.4, random_state=1).index
customers.loc[high_idx, "income"] = np.nan

# 2) 상품(products)
categories = ["패션", "뷰티", "식품", "가전", "도서"]
n_products = 40
products = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_products + 1)],
    "category": np.random.choice(categories, n_products),
    "price": np.random.choice([9900, 19900, 29900, 49900, 89900, 129900], n_products),
})

# 3) 주문(orders)
n_orders = 2000
order_customer = np.random.choice(customers["customer_id"], n_orders)
order_product = np.random.choice(products["product_id"], n_orders)
price_map = products.set_index("product_id")["price"]
quantity = np.random.choice([1, 1, 1, 2, 2, 3], n_orders)
amount = price_map.loc[order_product].values * quantity

orders = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_orders + 1)],
    "customer_id": order_customer,
    "product_id": order_product,
    "category": products.set_index("product_id").loc[order_product, "category"].values,
    "quantity": quantity,
    "amount": amount.astype(float),
    "channel": np.random.choice(["web", "app"], n_orders, p=[0.5, 0.5]),
})

# 채널별 금액 결측
app_mask = orders["channel"] == "app"
app_idx = orders[app_mask].sample(frac=0.04, random_state=2).index
web_idx = orders[~app_mask].sample(frac=0.01, random_state=3).index
orders.loc[app_idx.union(web_idx), "amount"] = np.nan

# 수량 및 금액 이상치
orders.loc[7, "quantity"] = 100
orders.loc[1900, "quantity"] = 50
orders.loc[1500, "amount"] = 5_000_000

print("모두마켓 데이터 생성 완료")
print(
    "customers:", customers.shape,
    "| products:", products.shape,
    "| orders:", orders.shape
)
print("\n[빠르게 둘러보기]")
print("customers 결측:", customers.isnull().sum().sum(), "건")
print("orders 결측:", orders.isnull().sum().sum(), "건")

모두마켓 데이터 생성 완료
customers: (300, 6) | products: (40, 3) | orders: (2000, 7)

[빠르게 둘러보기]
customers 결측: 51 건
orders 결측: 51 건


In [53]:
is_seoul = customers["region"] == "서울"
print("타입:", type(is_seoul).__name__, "/ True 개수:", is_seoul.sum())
print("앞 5개 미리보기:", is_seoul.head().tolist())

seoul_customers = customers[is_seoul]
print("\n서울 고객 수:", len(seoul_customers))
display(seoul_customers.head(3))

타입: Series / True 개수: 62
앞 5개 미리보기: [False, False, True, False, True]

서울 고객 수: 62


,customer_id,age,gender,region,membership,income
2,C0003,41,F,서울,premium,8500.0
4,C0005,33,M,서울,basic,2500.0
5,C0006,999,F,서울,basic,NaN


In [54]:
# 예제: isin과 between — 같은 결과를 두 방식으로
metro = ["서울", "경기", "인천"]

# 방법 A) 기본 |로 연결
mask_a = (customers["region"] == "서울") | (customers["region"] == "경기") | (customers["region"] == "인천")
print("방법 A 결과 수:", mask_a.sum())

# 방법 B) isin
mask_b = customers["region"].isin(metro)
print("방법 B 결과 수:", mask_b.sum())

# 두 결과가 완전히 같은지 검증
print("두 마스크가 완전히 동일?:", (mask_a == mask_b).all())

방법 A 결과 수: 176
방법 B 결과 수: 176
두 마스크가 완전히 동일?: True


In [55]:
# 예제: query — 문자열로 표현하는 필터
# 30대이면서 서울에 사는 프리미엄/VIP 고객
result = customers.query("age.between(30, 39) and region == '서울' and membership in ['premium', 'vip']")
print("해당 고객 수:", len(result))
display(result.head(3))

해당 고객 수: 4


,customer_id,age,gender,region,membership,income
25,C0026,36,M,서울,premium,2500.0
88,C0089,30,F,서울,premium,NaN
137,C0138,32,F,서울,premium,NaN


In [56]:
# 예제: ~ (NOT) — '서울 외 지역'
not_seoul = customers[~customers["region"].isin(["서울"])]
print("서울 외 지역 고객 수:", len(not_seoul))
print("지역 분포:")
print(not_seoul["region"].value_counts())

서울 외 지역 고객 수: 238
지역 분포:
region
부산    64
경기    60
대구    60
인천    54
Name: count, dtype: int64


In [57]:
# NaN과 비교는 항상 False 입니다 — 직관과 다름
sample = orders[["order_id", "amount"]].head(10).copy()
sample.loc[2, "amount"] = np.nan
display(sample)

print("\n== amount > 0 인 행 ==")
display(sample[sample["amount"] > 0])     # NaN 행은 제외됨 (조건이 False로 평가)

print("\n== amount <= 0 인 행 ==")
display(sample[sample["amount"] <= 0])    # 역시 NaN은 제외됨

,order_id,amount
0,O00001,29900.0
1,O00002,149700.0
2,O00003,NaN
3,O00004,29900.0
4,O00005,19800.0
5,O00006,129900.0
6,O00007,39800.0
7,O00008,29900.0
8,O00009,89900.0
9,O00010,9900.0



== amount > 0 인 행 ==


,order_id,amount
0,O00001,29900.0
1,O00002,149700.0
3,O00004,29900.0
4,O00005,19800.0
5,O00006,129900.0
6,O00007,39800.0
7,O00008,29900.0
8,O00009,89900.0
9,O00010,9900.0



== amount <= 0 인 행 ==


,order_id,amount


In [58]:
# 해결책 1) 결측을 명시적으로 포함하기
mask_with_na = (sample["amount"] > 0) | (sample["amount"].isnull())
print("결측 포함 필터:")
display(sample[mask_with_na])

# 해결책 2) 결측을 명시적으로 제외하기
mask_no_na = (sample["amount"] > 0) & (sample["amount"].notna())
print("\n결측 명시적 제외:")
display(sample[mask_no_na])

결측 포함 필터:


,order_id,amount
0,O00001,29900.0
1,O00002,149700.0
2,O00003,NaN
3,O00004,29900.0
4,O00005,19800.0
5,O00006,129900.0
6,O00007,39800.0
7,O00008,29900.0
8,O00009,89900.0
9,O00010,9900.0



결측 명시적 제외:


,order_id,amount
0,O00001,29900.0
1,O00002,149700.0
3,O00004,29900.0
4,O00005,19800.0
5,O00006,129900.0
6,O00007,39800.0
7,O00008,29900.0
8,O00009,89900.0
9,O00010,9900.0


In [59]:
# 스스로 해보자! (2)
# 1) isin + notna 조합
# mask = orders["category"].isin([___, ___]) & orders["amount"].___()
# print("건수:", mask.sum())

# 2) query로 같은 조건
# print(orders.query("category in [___, ___] and amount.notna()").shape)

In [60]:
# 예제: isin과 between — 같은 결과를 두 방식으로
metro = ["서울", "경기", "인천"]

# 방법 A) 기본 |로 연결
mask_a = (customers["region"] == "서울") | (customers["region"] == "경기") | (customers["region"] == "인천")
print("방법 A 결과 수:", mask_a.sum())

# 방법 B) isin
mask_b = customers["region"].isin(metro)
print("방법 B 결과 수:", mask_b.sum())

# 두 결과가 완전히 같은지 검증
print("두 마스크가 완전히 동일?:", (mask_a == mask_b).all())

방법 A 결과 수: 176
방법 B 결과 수: 176
두 마스크가 완전히 동일?: True


In [61]:
# 예제: query — 문자열로 표현하는 필터
# 30대이면서 서울에 사는 프리미엄/VIP 고객
result = customers.query("age.between(30, 39) and region == '서울' and membership in ['premium', 'vip']")
print("해당 고객 수:", len(result))
display(result.head(3))

해당 고객 수: 4


,customer_id,age,gender,region,membership,income
25,C0026,36,M,서울,premium,2500.0
88,C0089,30,F,서울,premium,NaN
137,C0138,32,F,서울,premium,NaN


import pandas as pd

data = {
    "이름": ["철수", "영희", "민수"],
    "점수": [85, 92, 78]
}
df = pd.DataFrame(data)
df

In [62]:
cd ..
git add .
git commit -m "add: 마크다운/노트북 실습 파일 추가"
git push

SyntaxError: invalid syntax (1076960983.py, line 1)